In [1]:
# to import things from script folder on this notebooks
import sys; sys.path.append('..')
from Bio import AlignIO, SeqIO
import pandas as pd
from pathlib import Path
import itertools as it
import tempfile


from src.conf import DATA_RAW, DATA_INTERIM, DATA_PROCESSED, protein_names
from src._Mafft import MafftCommandline


In [2]:
# others proteins
sars2protein_folder = DATA_INTERIM/'sars2_proteins'


sars2_splits = DATA_INTERIM/'sars2_splits'
sars2_country = DATA_INTERIM/'sars2_country'

mafft_out = DATA_PROCESSED/"mafftout"
if not mafft_out.is_dir():
    mafft_out.mkdir()
    

## Align country filtered seq

In [14]:
#keyfunc = lambda path: path.name[:-6].split("_")[3]
c = 'China'
prot_country = sars2_country.glob(f'*{c}.fasta')


In [15]:
for pfile in prot_country:
    pname = pfile.name[:-6].split('_')[2]
    # reference sq, from wuhan
    refproteinseq = sars2protein_folder/f'sars2_refWuhan_{pname}.fasta'
    
    # Run mafft ali
    outali = mafft_out/f'{pname}_mafft_{c}.ali'
    mafft_cline = MafftCommandline(input=refproteinseq, addfragments=pfile, auto=True,
                                   thread=10, keeplength=True,reorder=True)
    print(mafft_cline)
    # Ejecutando Mafft
    stdout, stderr = mafft_cline()
    with open(outali,"w") as handle:
        handle.write(stdout)

mafft --auto --thread 10 --keeplength --reorder --addfragments /mnt/d/Python_projects/sars-cov-2_proteome/data/interim/sars2_country/sars2_others_E_China.fasta /mnt/d/Python_projects/sars-cov-2_proteome/data/interim/sars2_proteins/sars2_refWuhan_E.fasta
mafft --auto --thread 10 --keeplength --reorder --addfragments /mnt/d/Python_projects/sars-cov-2_proteome/data/interim/sars2_country/sars2_others_M_China.fasta /mnt/d/Python_projects/sars-cov-2_proteome/data/interim/sars2_proteins/sars2_refWuhan_M.fasta
mafft --auto --thread 10 --keeplength --reorder --addfragments /mnt/d/Python_projects/sars-cov-2_proteome/data/interim/sars2_country/sars2_others_NS3_China.fasta /mnt/d/Python_projects/sars-cov-2_proteome/data/interim/sars2_proteins/sars2_refWuhan_NS3.fasta
mafft --auto --thread 10 --keeplength --reorder --addfragments /mnt/d/Python_projects/sars-cov-2_proteome/data/interim/sars2_country/sars2_others_NS6_China.fasta /mnt/d/Python_projects/sars-cov-2_proteome/data/interim/sars2_proteins/s

## align splits

In [35]:
keyfunc = lambda path: path.name.split("_")[2]
prot_split_groups = it.groupby(sars2_splits.glob('*.fasta'), keyfunc)


In [12]:

for pname, psplit in prot_split_groups:
    
    # reference sq, from wuhan
    refproteinseq = sars2protein_folder/f'sars2_refWuhan_{pname}.fasta'
    #mafft_prot_out = 'mafft'
    
    for i,sp in enumerate(psplit,1):
        
        
        outali = mafft_out/f'{pname}_mafft_refsplit{i}.ali'
        mafft_cline = MafftCommandline(input=refproteinseq, addfragments=sp, auto=True,
                                       thread=10, keeplength=True)
        print(mafft_cline)
        break
        # Ejecutando Mafft
        stdout, stderr = mafft_cline()
        with open(outali,"w") as handle:
            handle.write(stdout)
        refproteinseq = outali
        
    break
    

mafft --auto --thread 10 --keeplength --addfragments /mnt/d/Python_projects/sars-cov-2_proteome/data/interim/sars2_splits/sars2_others_M_split1.fasta /mnt/d/Python_projects/sars-cov-2_proteome/data/interim/sars2_proteins/sars2_refWuhan_M.fasta


## testing GISAID method, this is based on nucleotid seq

This full alignment (msa_0922.fasta) is based on 105,360 submissions to EpiCoV.
Both duplicate and low-quality sequences (>5% NNNNs) have been removed, using only complete sequences (length >29,000 bp). The resulting alignment of 97,686 sequences is created using mafft https://doi.org/10.1093/molbev/mst010 in 3 separate steps.

(1) Each sequence is individually aligned to the reference hCoV-19/Wuhan/WIV04/2019 (EPI_ISL_402124). Sequences that created dubious insertions of >=30 nucleotides in the reference sequence and occurred only once in the database are discarded. The alignments are created with the command:
mafft --thread -1 input.fasta > output.fasta

(2) All sequences that result in insertions in the reference from (1) are then aligned with a opening gap penalty of 10 to prevent long stretches of dubious insertions in the alignment due to the presence of long stretches of NNNNs. The following command is used:
mafft --retree 3 --maxiterate 10 --thread -1 --nomemsave --op 10 seqsCausingInsertionsInRef.fasta > seqsCausingInsertionsInRef_aligned.fasta

(3) The rest of the sequences that did not result in insertions are aligned to the resulting alignment in step (2) with this command:
mafft --thread 1 --quiet --keeplength --add sequencesNotCausingInsertionsInRef.fa seqsCausingInsertionsInRef_aligned.fasta > msa_0922.fasta

Thanks to Rob Lanfear for discussion and suggestions.

In [41]:
others_groups = sars2protein_folder.glob('sars2_others*')


In [71]:
mafft_out = DATA_PROCESSED/"mafftout"
if not mafft_out.is_dir():
    mafft_out.mkdir()
    
for pfile in sars2protein_folder.glob('sars2_others*'):
    
    pname = pfile.name[:-6].split("_")[2]
    # parse other proteins
    records = SeqIO.parse(pfile,"fasta")
    
    # reference seq, from wuhan
    refproteinseq = sars2protein_folder/f'sars2_refWuhan_{pname}.fasta'
    # read ref
    ref_record = SeqIO.read(refproteinseq,"fasta")
    mafft_prot_out = mafft_out/f"{pname}"
    
    # create mafft_prot dir
    if not mafft_prot_out.is_dir():
        mafft_prot_out.mkdir()
    # METHOD - step 1)
    for i,rec in enumerate(records,1):
        
        # write tmp file to run mafft for each individual file
        with tempfile.NamedTemporaryFile() as fp:
            fp.write(f'>{ref_record.description}\n'.encode())
            fp.write(f'{ref_record.seq}\n'.encode())
            
            fp.write(f'>{rec.description}\n'.encode())
            fp.write(f'{rec.seq}\n'.encode())
            fp.seek(0)
            fp.read()
            print(fp)
            
            outali = mafft_prot_out/f'{pname}_mafft_seq{i}.ali'
            mafft_cline = MafftCommandline(input=fp.name, thread=10)
            print(mafft_cline)
            
            # Ejecutando Mafft
            stdout, stderr = mafft_cline()
            with open(outali,"w") as handle:
                handle.write(stdout)
    
    break
        
    

mafft --thread 10 /tmp/tmp41d2esef
mafft --thread 10 /tmp/tmpf97dcpou
mafft --thread 10 /tmp/tmpy9fqselm
mafft --thread 10 /tmp/tmpwop8tzn6
mafft --thread 10 /tmp/tmp8s759n7i
mafft --thread 10 /tmp/tmpky9b_yx1
mafft --thread 10 /tmp/tmp9w0t0g3k
mafft --thread 10 /tmp/tmpugxhl6rz
mafft --thread 10 /tmp/tmpoa0fvs79
mafft --thread 10 /tmp/tmpmg_mpvqg
mafft --thread 10 /tmp/tmpgg7ffqfr
mafft --thread 10 /tmp/tmpz0axvcqs
mafft --thread 10 /tmp/tmp0f7p4avd
mafft --thread 10 /tmp/tmpkacvrbjd
mafft --thread 10 /tmp/tmprlb_09q1
mafft --thread 10 /tmp/tmp19664jw6
mafft --thread 10 /tmp/tmp1ldh6d9j
mafft --thread 10 /tmp/tmp2i6ny7ks
mafft --thread 10 /tmp/tmphh_90b5u
mafft --thread 10 /tmp/tmp09w4l431
mafft --thread 10 /tmp/tmpz8xo_7_i
mafft --thread 10 /tmp/tmpyr5suig0
mafft --thread 10 /tmp/tmpucmfzp9z
mafft --thread 10 /tmp/tmp56lnt9qr
mafft --thread 10 /tmp/tmpo5p8xiwq
mafft --thread 10 /tmp/tmpc5lbvr0j
mafft --thread 10 /tmp/tmpxzdy84mx
mafft --thread 10 /tmp/tmp42q4innc
mafft --thread 10 /t

mafft --thread 10 /tmp/tmp_40dnzb0
mafft --thread 10 /tmp/tmp8rmhmhn4
mafft --thread 10 /tmp/tmp8s8bany9
mafft --thread 10 /tmp/tmp9cevyjos


KeyboardInterrupt: 